# Lab 2.1 (ต่อ) · Data Profiling ของ `customers.csv`

**ต่อจาก "เสร็จเร็ว? ลองต่อ" ของ Lab 2.1** · ใช้ prompt แนวเดิม: **รายงานก่อน ยังไม่แก้**

`customers.csv` คือข้อมูลสมาชิก 1 แถว = ลูกค้า 1 คน เชื่อมกับยอดขายด้วย `customer_id`
ไฟล์นี้ **ดูสะอาดเมื่อมองผ่าน ๆ** (ไม่มีค่าว่าง ไม่มีแถวซ้ำ) ปัญหาจึงไม่ได้อยู่ที่รูปแบบ แต่อยู่ที่ **ความหมาย** และ **การเชื่อมกับตารางอื่น**

| ขั้น | ทำอะไร |
|---|---|
| 0 | เตรียมเครื่องมือ (`check()`) |
| 1 | โหลดและดูข้อมูลครั้งแรก |
| 2 | Profiling พื้นฐาน: ค่าว่าง ค่าไม่ซ้ำ แถวซ้ำ รูปแบบ |
| 3 | ตรวจค่าที่อนุญาต (domain) และช่วงวันที่ |
| 4 | ตรวจความสอดคล้องกับ `sales_clean.csv` (cross-table) |
| 5 | ตรวจคำตอบด้วย `check()` |
| 6 | สรุปความผิดปกติ และตัดสินใจ (ยังไม่แก้) |

ไฟล์ที่ต้องอัปโหลด: `customers.csv` และ `sales_clean.csv` (ผลจาก Lab 2.1 ขั้นที่ 6)
ไฟล์เสริม (ถ้ามี): `branches.xlsx` หรือ `branches.csv` ตารางรหัสสาขาทางการ อยู่ใน `Lab2/lab2.2-dashboard/public/`

## ขั้นที่ 0 · เตรียมเครื่องมือ
รันเซลล์นี้ก่อน (ไม่ต้องแก้) มีฟังก์ชัน `load_csv()` และ `check()` สำหรับตรวจคำตอบโดยไม่เฉลยตัวเลข

In [1]:
# ===== เซลล์ตัวช่วย · รันก่อน ไม่ต้องแก้ =====
import base64, json, os, re
import pandas as pd

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 140)

_EXP = json.loads(base64.b64decode("eyJyb3dzIjogMzAwMCwgImR1cF9yb3dzIjogMCwgImR1cF9pZCI6IDAsICJiYWRfaWQiOiAwLCAiYmFkX2RhdGUiOiAwLCAiYmFkX3Bob25lIjogMCwgInNoYXJlZF9waG9uZSI6IDk5LCAib3JwaGFuX3NhbGVzIjogMCwgIm5ldmVyX2JvdWdodCI6IDQ5MywgImpvaW5lZF9hZnRlcl9idXkiOiAwLCAiaG9tZV9taXNtYXRjaCI6IDExNSwgIm1pbm9yIjogMTEzLCAiam9pbmVkX2JlZm9yZV9vcGVuIjogMH0=").decode())
_LABEL = {
    "rows": "จำนวนแถวทั้งหมด",
    "dup_rows": "แถวที่ซ้ำกันทุกคอลัมน์",
    "dup_id": "customer_id ที่ซ้ำ",
    "bad_id": "customer_id ที่ไม่ใช่รูปแบบ C + ตัวเลข 5 หลัก",
    "bad_date": "joined_date ที่ไม่ใช่วันที่ YYYY-MM-DD ที่มีจริง",
    "bad_phone": "phone ที่ไม่ใช่รูปแบบ 0XX-xxx-XXXX",
    "shared_phone": "แถวที่ใช้เบอร์โทร (แบบปิดบัง) ซ้ำกับคนอื่น",
    "orphan_sales": "customer_id ใน sales ที่ไม่มีใน customers",
    "never_bought": "สมาชิกที่ไม่เคยซื้อเลย",
    "joined_after_buy": "สมาชิกที่วันสมัครอยู่หลังวันซื้อครั้งแรก",
    "home_mismatch": "สมาชิกที่สาขาประจำ ≠ สาขาที่ซื้อบ่อยสุด",
    "minor": "สมาชิกอายุต่ำกว่า 18",
    "joined_before_open": "สมาชิกที่สมัครก่อนสาขาประจำเปิด",
}
_HINT = {
    "dup_rows": "ใช้ df.duplicated() ทั้งแถว",
    "dup_id": "ใช้ df['customer_id'].duplicated()",
    "shared_phone": "นับทุกแถวที่เบอร์ซ้ำ ใช้ duplicated(keep=False)",
    "orphan_sales": "ตัด customer_id ว่าง (ลูกค้าทั่วไป) ออกก่อน",
    "never_bought": "ลูกค้าใน customers ที่ไม่มี customer_id นี้ใน sales เลย",
    "joined_after_buy": "เทียบ joined_date กับ 10 ตัวแรกของ datetime ที่น้อยที่สุดของลูกค้าคนนั้น",
    "home_mismatch": "ต้องแปลงรหัส B01–B05 เป็นชื่อสาขาก่อน แล้วเทียบกับสาขาที่ลูกค้าซื้อบ่อยสุด",
}

def check(name, value):
    """เทียบคำตอบกับเฉลยโดยไม่บอกตัวเลข"""
    exp, label = _EXP[name], _LABEL[name]
    if value is None:
        print(f"⬜ {label}: ยังไม่ได้ใส่ค่า")
    elif int(value) == exp:
        print(f"✅ {label}: {int(value):,}")
    elif abs(int(value) - exp) <= max(2, exp * 0.03):
        print(f"🟡 {label}: {int(value):,} ใกล้แล้ว · {_HINT.get(name, 'ตรวจเงื่อนไขอีกครั้ง')}")
    else:
        print(f"❌ {label}: {int(value):,} ยังไม่ถูก · {_HINT.get(name, 'ตรวจเงื่อนไขอีกครั้ง')}")

def load_csv(name):
    """อ่าน CSV ทุกคอลัมน์เป็นข้อความ ค่าว่างเป็น "" ตัด BOM · ถ้าไม่มีไฟล์จะให้อัปโหลด (Colab)"""
    if not os.path.exists(name):
        try:
            from google.colab import files
            print(f"ไม่พบ {name} กรุณาอัปโหลดไฟล์")
            files.upload()
        except ImportError:
            raise FileNotFoundError(f"ไม่พบไฟล์ {name} ให้วางไฟล์ไว้โฟลเดอร์เดียวกับ notebook")
    return pd.read_csv(name, dtype=str, keep_default_na=False, encoding="utf-8-sig")

def section(title):
    print(f"\n{'=' * 10} {title} {'=' * 10}")

## ขั้นที่ 1 · ดูข้อมูลครั้งแรก
อ่าน **ทุกคอลัมน์เป็นข้อความ** และใช้ `encoding="utf-8-sig"` เพราะไฟล์นี้มี BOM (ถ้าไม่ตัด ชื่อคอลัมน์แรกจะกลายเป็น `\ufeffcustomer_id`)

In [2]:
cust = load_csv("customers.csv")
sales = load_csv("sales_clean.csv")
print(f"customers: {len(cust):,} แถว × {cust.shape[1]} คอลัมน์ → {list(cust.columns)}")
print(f"sales_clean: {len(sales):,} แถว")
cust.head(8)

customers: 3,000 แถว × 7 คอลัมน์ → ['customer_id', 'nickname', 'gender', 'age_group', 'home_branch_id', 'joined_date', 'phone']
sales_clean: 53,057 แถว


,customer_id,nickname,gender,age_group,home_branch_id,joined_date,phone
0,C00001,กมล,หญิง,18-24,B02,2026-03-03,082-xxx-6693
1,C00002,กิตติ,ชาย,25-34,B04,2026-01-18,084-xxx-3734
2,C00003,ปิยะ,หญิง,18-24,B04,2025-05-04,089-xxx-9286
3,C00004,ปิยะ,หญิง,18-24,B05,2026-05-05,083-xxx-7314
4,C00005,รัตนา,หญิง,25-34,B04,2026-07-10,083-xxx-5701
5,C00006,กมล,ชาย,18-24,B04,2026-08-16,086-xxx-1895
6,C00007,วรรณ,ชาย,45-54,B03,2026-07-17,087-xxx-3147
7,C00008,ภัทร,หญิง,25-34,B01,2026-04-07,087-xxx-9877


**คำถามชวนคิด:** คอลัมน์ไหนน่าจะใช้เป็น "กุญแจ" เชื่อมกับตาราง sales ได้ และคอลัมน์ไหน **ไม่ควร** ใช้ระบุตัวคน

## ขั้นที่ 2 · Profiling พื้นฐาน (รายงานอย่างเดียว)

**Prompt**
```
ช่วยทำ data profiling ของ DataFrame cust ด้วย pandas (ทุกคอลัมน์เป็น str ค่าว่างเป็น "")
- จำนวนค่าว่าง (รวมค่าที่มีแต่ช่องว่าง) ค่ามีช่องว่างหัว/ท้าย และค่าไม่ซ้ำของแต่ละคอลัมน์
- จำนวนแถวที่ซ้ำกันทุกคอลัมน์ และ customer_id ที่ซ้ำ
- customer_id ที่ไม่ใช่รูปแบบ C ตามด้วยตัวเลข 5 หลัก
- joined_date ที่ไม่ใช่ YYYY-MM-DD หรือไม่ใช่วันที่มีจริง
- phone ที่ไม่ใช่รูปแบบ 0XX-xxx-XXXX และจำนวนแถวที่เบอร์ซ้ำกัน
เก็บจำนวนไว้ในตัวแปร n_dup_rows, n_dup_id, n_bad_id, n_bad_date, n_bad_phone, n_shared_phone
อย่าแก้ข้อมูล ให้รายงานอย่างเดียว
```

In [3]:
section("1) ค่าว่าง / ช่องว่างหัว-ท้าย / ค่าไม่ซ้ำ")
profile = pd.DataFrame({
    "ค่าว่าง": cust.apply(lambda s: s.str.strip().eq("")).sum(),
    "มีช่องว่างหัว/ท้าย": cust.apply(lambda s: s.ne(s.str.strip())).sum(),
    "ค่าไม่ซ้ำ": cust.nunique(),
})
print(profile.to_string())

section("2) แถวซ้ำ / customer_id ซ้ำ")
n_dup_rows = int(cust.duplicated().sum())
n_dup_id = int(cust["customer_id"].duplicated().sum())
print(f"n_dup_rows = {n_dup_rows:,} · n_dup_id = {n_dup_id:,}")

section("3) รูปแบบ customer_id")
bad_id = ~cust["customer_id"].str.fullmatch(r"C\d{5}")
n_bad_id = int(bad_id.sum())
print(f"n_bad_id = {n_bad_id:,}", cust.loc[bad_id, "customer_id"].head().tolist())

section("4) joined_date")
date_fmt = cust["joined_date"].str.fullmatch(r"\d{4}-\d{2}-\d{2}")
joined = pd.to_datetime(cust["joined_date"].where(date_fmt), format="%Y-%m-%d", errors="coerce")
bad_date = joined.isna()
n_bad_date = int(bad_date.sum())
print(f"n_bad_date = {n_bad_date:,}", cust.loc[bad_date, "joined_date"].head().tolist())
print(f"ช่วงวันที่สมัคร: {joined.min():%Y-%m-%d} ถึง {joined.max():%Y-%m-%d}")

section("5) phone")
bad_phone = ~cust["phone"].str.fullmatch(r"0\d{2}-xxx-\d{4}")
n_bad_phone = int(bad_phone.sum())
shared = cust["phone"].duplicated(keep=False)
n_shared_phone = int(shared.sum())
print(f"n_bad_phone = {n_bad_phone:,} · n_shared_phone = {n_shared_phone:,} แถว ({cust.loc[shared, 'phone'].nunique():,} เบอร์)")
print(cust[shared].sort_values("phone").head(6).to_string())


========== 1) ค่าว่าง / ช่องว่างหัว-ท้าย / ค่าไม่ซ้ำ ==========
                ค่าว่าง  มีช่องว่างหัว/ท้าย  ค่าไม่ซ้ำ
customer_id           0                   0       3000
nickname              0                   0         24
gender                0                   0          3
age_group             0                   0          6
home_branch_id        0                   0          5
joined_date           0                   0        530
phone                 0                   0       2950

========== 2) แถวซ้ำ / customer_id ซ้ำ ==========
n_dup_rows = 0 · n_dup_id = 0

========== 3) รูปแบบ customer_id ==========
n_bad_id = 0 []

========== 4) joined_date ==========
n_bad_date = 0 []
ช่วงวันที่สมัคร: 2025-04-01 ถึง 2026-09-14

========== 5) phone ==========
n_bad_phone = 0 · n_shared_phone = 99 แถว (49 เบอร์)
     customer_id nickname gender age_group home_branch_id joined_date         phone
2206      C02207    ทิพย์    ชาย     35-44            B02  2025-04-14  081-xxx-4308
2

**ลองคิด:** เบอร์ที่ซ้ำกันเป็นลูกค้าคนเดียวกันที่สมัครซ้ำ หรือเป็นเรื่องบังเอิญ
เบอร์ถูกปิดบังเหลือแค่ **prefix 3 หลัก + 4 หลักท้าย** จึงมีรูปแบบที่เป็นไปได้ราว 9 × 10,000 = 90,000 แบบ
เซลล์ถัดไปคำนวณว่า "ถ้าสุ่มล้วน ๆ" ควรชนกันกี่คู่ (birthday problem) และดูว่าคนที่เบอร์ซ้ำมีข้อมูลอื่นเหมือนกันไหม

In [4]:
n = len(cust)
space = cust["phone"].str[:3].nunique() * 10_000
expected_pairs = n * (n - 1) / (2 * space)
groups = cust[shared].groupby("phone")
same_profile = groups.apply(lambda g: g[["nickname", "gender", "age_group"]].nunique().max() == 1, include_groups=False).sum()
print(f"คู่ที่ชนกันจริง: {cust.loc[shared, 'phone'].nunique():,} คู่ · ถ้าสุ่มล้วน ๆ คาดว่า ≈ {expected_pairs:.0f} คู่")
print(f"กลุ่มที่ชื่อเล่น+เพศ+ช่วงอายุ เหมือนกันทั้งหมด: {same_profile} กลุ่ม")
print("→ สรุป: ถ้าจำนวนใกล้ค่าที่คาด และข้อมูลอื่นไม่ตรงกัน แปลว่าเป็นการชนกันเพราะปิดบังเบอร์ ไม่ใช่สมาชิกซ้ำ")

คู่ที่ชนกันจริง: 49 คู่ · ถ้าสุ่มล้วน ๆ คาดว่า ≈ 50 คู่
กลุ่มที่ชื่อเล่น+เพศ+ช่วงอายุ เหมือนกันทั้งหมด: 0 กลุ่ม
→ สรุป: ถ้าจำนวนใกล้ค่าที่คาด และข้อมูลอื่นไม่ตรงกัน แปลว่าเป็นการชนกันเพราะปิดบังเบอร์ ไม่ใช่สมาชิกซ้ำ


## ขั้นที่ 3 · ตรวจค่าที่อนุญาต (domain) และช่วงวันที่

**Prompt**
```
แสดง value_counts (ใช้ repr ให้เห็นช่องว่าง) ของ gender, age_group, home_branch_id
ตรวจว่ามีค่านอกรายการที่อนุญาตไหม และ joined_date อยู่ในช่วงที่ร้านเปิดขาย (ตามข้อมูล sales) หรือไม่
เก็บจำนวนสมาชิกอายุต่ำกว่า 18 ไว้ใน n_minor
```

In [5]:
ALLOWED = {
    "gender": {"หญิง", "ชาย", "ไม่ระบุ"},
    "age_group": {"ต่ำกว่า 18", "18-24", "25-34", "35-44", "45-54", "55+"},
    "home_branch_id": {"B01", "B02", "B03", "B04", "B05"},
}
for col, allowed in ALLOWED.items():
    section(col)
    vc = cust[col].value_counts()
    for v, c in vc.items():
        print(f"{v!r:<14} {c:>6,}" + ("" if v in allowed else "   <-- นอกรายการ"))
    print(f"ค่านอกรายการ: {int((~cust[col].isin(allowed)).sum()):,}")

n_minor = int((cust["age_group"] == "ต่ำกว่า 18").sum())
print(f"\nn_minor = {n_minor:,}  (ต้องระวังเรื่อง PDPA: ข้อมูลผู้เยาว์)")

section("joined_date เทียบกับช่วงที่มียอดขาย")
sale_start, sale_end = sales["datetime"].str[:10].min(), sales["datetime"].str[:10].max()
print(f"ยอดขาย: {sale_start} ถึง {sale_end}")
print(f"สมัครก่อนเปิดร้าน: {int((cust['joined_date'] < sale_start).sum()):,} · สมัครหลังข้อมูลล่าสุด: {int((cust['joined_date'] > sale_end).sum()):,}")


========== gender ==========
'หญิง'          1,652
'ชาย'           1,232
'ไม่ระบุ'         116
ค่านอกรายการ: 0

========== age_group ==========
'25-34'           993
'18-24'           775
'35-44'           577
'45-54'           346
'55+'             196
'ต่ำกว่า 18'      113
ค่านอกรายการ: 0

========== home_branch_id ==========
'B01'             776
'B05'             693
'B02'             643
'B04'             483
'B03'             405
ค่านอกรายการ: 0

n_minor = 113  (ต้องระวังเรื่อง PDPA: ข้อมูลผู้เยาว์)

========== joined_date เทียบกับช่วงที่มียอดขาย ==========


ยอดขาย: 2025-04-01 ถึง 2026-09-20
สมัครก่อนเปิดร้าน: 0 · สมัครหลังข้อมูลล่าสุด: 0


## ขั้นที่ 4 · ตรวจความสอดคล้องกับ `sales_clean.csv`

ปัญหาที่หาไม่เจอถ้าดูตารางเดียว ต้องเอา 2 ตารางมาเทียบกัน

**Prompt**
```
เทียบ cust กับ sales (customer_id ว่างใน sales = ลูกค้าทั่วไป ไม่นับ)
- customer_id ใน sales ที่ไม่มีใน cust (n_orphan_sales)
- สมาชิกที่ไม่เคยซื้อเลย (n_never_bought) และในนั้นกี่คนที่เพิ่งสมัคร 30 วันล่าสุด
- สมาชิกที่ joined_date อยู่หลังวันที่ซื้อครั้งแรก (n_joined_after_buy)
- home_branch_id เป็นรหัส B01–B05 แต่ sales ใช้ชื่อสาขา: หา mapping ที่น่าจะถูกจากสาขาที่สมาชิกซื้อบ่อยสุด
  แล้วนับสมาชิกที่สาขาประจำไม่ตรงกับสาขาที่ซื้อบ่อยสุด (n_home_mismatch)
- สมาชิกสาขา B03 (อารีย์) ที่สมัครก่อนสาขาอารีย์เปิดขายวันแรก
```

In [6]:
members = sales[sales["customer_id"] != ""]
sale_ids, cust_ids = set(members["customer_id"]), set(cust["customer_id"])

section("1) customer_id ใน sales ที่ไม่มีใน customers")
orphan = sorted(sale_ids - cust_ids)
n_orphan_sales = len(orphan)
print(f"n_orphan_sales = {n_orphan_sales:,}", orphan[:5])

section("2) สมาชิกที่ไม่เคยซื้อ")
never = cust[~cust["customer_id"].isin(sale_ids)]
n_never_bought = len(never)
recent = int((never["joined_date"] > (pd.Timestamp(sale_end) - pd.Timedelta(days=30)).strftime("%Y-%m-%d")).sum())
print(f"n_never_bought = {n_never_bought:,} ({n_never_bought / len(cust):.1%} ของสมาชิก) · ในนั้นสมัครใน 30 วันล่าสุด {recent:,} คน")

section("3) วันสมัครอยู่หลังวันซื้อครั้งแรก")
first_buy = members.groupby("customer_id")["datetime"].min().str[:10]
j = cust.set_index("customer_id")["joined_date"].reindex(first_buy.index)
after = j > first_buy
n_joined_after_buy = int(after.sum())
print(f"n_joined_after_buy = {n_joined_after_buy:,}")

section("4) home_branch_id (รหัส) ↔ ชื่อสาขาใน sales")
fav = members.groupby("customer_id")["branch"].agg(lambda s: s.value_counts().idxmax())
home = cust.set_index("customer_id")["home_branch_id"].reindex(fav.index)
xtab = pd.crosstab(home, fav)
print(xtab.to_string())
BRANCH_CODE = xtab.idxmax(axis=1).to_dict()          # รหัส → ชื่อสาขาที่พบบ่อยสุด
agree = xtab.max(axis=1) / xtab.sum(axis=1)
print("\nmapping ที่อนุมานได้:", {k: f"{v} ({agree[k]:.0%})" for k, v in BRANCH_CODE.items()})
n_home_mismatch = int((home.map(BRANCH_CODE) != fav).sum())
print(f"n_home_mismatch = {n_home_mismatch:,} จาก {len(fav):,} คนที่เคยซื้อ ({n_home_mismatch / len(fav):.1%})")

section("5) เทียบ mapping กับตารางสาขาทางการ (branches.xlsx / branches.csv)")
branches = None
for fname in ["branches.csv", "branches.xlsx"]:
    if os.path.exists(fname):
        branches = (pd.read_csv(fname, dtype=str, keep_default_na=False, encoding="utf-8-sig") if fname.endswith(".csv")
                    else pd.read_excel(fname, dtype=str).fillna(""))
        print(f"ใช้ไฟล์ {fname}")
        break
if branches is None:
    print("ไม่พบ branches.csv / branches.xlsx → ใช้ mapping ที่อนุมานจากข้อมูลไปก่อน (อัปโหลดไฟล์แล้วรันเซลล์นี้ใหม่ได้)")
    official = BRANCH_CODE
    opened = sales.groupby("branch")["datetime"].min().str[:10].to_dict()   # วันแรกที่มียอดขาย
else:
    official = dict(zip(branches["branch_id"], branches["branch"]))
    opened = dict(zip(branches["branch"], branches["opened_date"].str[:10]))
    diff = {k: (v, official.get(k)) for k, v in BRANCH_CODE.items() if official.get(k) != v}
    print("mapping ที่อนุมาน ตรงกับตารางทางการทุกรหัส ✅" if not diff else f"ไม่ตรง ❌ {diff}")
    BRANCH_CODE = official                                                   # ใช้ของทางการต่อจากนี้

section("6) สมาชิกที่สมัครก่อนสาขาประจำเปิด")
home_name = cust["home_branch_id"].map(BRANCH_CODE)
open_date = home_name.map(opened)
n_joined_before_open = int((cust["joined_date"] < open_date).sum())
for code, name in BRANCH_CODE.items():
    print(f"{code} {name:<12} เปิด {opened.get(name, '-')}")
print(f"n_joined_before_open = {n_joined_before_open:,}")


========== 1) customer_id ใน sales ที่ไม่มีใน customers ==========
n_orphan_sales = 0 []

========== 2) สมาชิกที่ไม่เคยซื้อ ==========
n_never_bought = 493 (16.4% ของสมาชิก) · ในนั้นสมัครใน 30 วันล่าสุด 97 คน

========== 3) วันสมัครอยู่หลังวันซื้อครั้งแรก ==========


n_joined_after_buy = 0

========== 4) home_branch_id (รหัส) ↔ ชื่อสาขาใน sales ==========


branch          บางนา  มหาวิทยาลัย  สยาม  สีลม  อารีย์
home_branch_id                                        
B01                 8            4   639     5       3
B02                 6            3    13   508       8
B03                 1            5     3     1     336
B04               403            4     3     5       5
B05                 5          506    16    13       4

mapping ที่อนุมานได้: {'B01': 'สยาม (97%)', 'B02': 'สีลม (94%)', 'B03': 'อารีย์ (97%)', 'B04': 'บางนา (96%)', 'B05': 'มหาวิทยาลัย (93%)'}
n_home_mismatch = 115 จาก 2,507 คนที่เคยซื้อ (4.6%)

========== 5) เทียบ mapping กับตารางสาขาทางการ (branches.xlsx / branches.csv) ==========
ใช้ไฟล์ branches.xlsx
mapping ที่อนุมาน ตรงกับตารางทางการทุกรหัส ✅

========== 6) สมาชิกที่สมัครก่อนสาขาประจำเปิด ==========
B01 สยาม         เปิด 2023-06-01
B02 สีลม         เปิด 2023-09-15
B03 อารีย์       เปิด 2025-11-01
B04 บางนา        เปิด 2024-03-01
B05 มหาวิทยาลัย  เปิด 2024-06-01
n_joined_before_open = 0


## ขั้นที่ 5 · ตรวจคำตอบ
ได้ ✅ ครบแล้วค่อยไปขั้นต่อไป ถ้าได้ 🟡 หรือ ❌ ให้อ่านคำใบ้แล้วสั่ง AI แก้โค้ด

In [7]:
check("rows", len(cust))
check("dup_rows", n_dup_rows)
check("dup_id", n_dup_id)
check("bad_id", n_bad_id)
check("bad_date", n_bad_date)
check("bad_phone", n_bad_phone)
check("shared_phone", n_shared_phone)
check("orphan_sales", n_orphan_sales)
check("never_bought", n_never_bought)
check("joined_after_buy", n_joined_after_buy)
check("home_mismatch", n_home_mismatch)
check("minor", n_minor)
check("joined_before_open", n_joined_before_open)

✅ จำนวนแถวทั้งหมด: 3,000
✅ แถวที่ซ้ำกันทุกคอลัมน์: 0
✅ customer_id ที่ซ้ำ: 0
✅ customer_id ที่ไม่ใช่รูปแบบ C + ตัวเลข 5 หลัก: 0
✅ joined_date ที่ไม่ใช่วันที่ YYYY-MM-DD ที่มีจริง: 0
✅ phone ที่ไม่ใช่รูปแบบ 0XX-xxx-XXXX: 0
✅ แถวที่ใช้เบอร์โทร (แบบปิดบัง) ซ้ำกับคนอื่น: 99
✅ customer_id ใน sales ที่ไม่มีใน customers: 0
✅ สมาชิกที่ไม่เคยซื้อเลย: 493
✅ สมาชิกที่วันสมัครอยู่หลังวันซื้อครั้งแรก: 0
✅ สมาชิกที่สาขาประจำ ≠ สาขาที่ซื้อบ่อยสุด: 115
✅ สมาชิกอายุต่ำกว่า 18: 113
✅ สมาชิกที่สมัครก่อนสาขาประจำเปิด: 0


## ขั้นที่ 6 · สรุปความผิดปกติ และตัดสินใจ (ยังไม่แก้)

เซลล์ถัดไปรวมผลทั้งหมดเป็นตาราง แล้วบันทึกเป็น `customers_profile.csv` ไว้แนบใน repo คู่กับ `cleaning_log.csv`

ดับเบิลคลิกเซลล์นี้เพื่อกรอกการตัดสินใจ

ตารางนี้กรอก **คำแนะนำ** ไว้แล้วจากผลด้านบน แก้ได้ถ้าคุณตัดสินใจต่าง

| ปัญหา | ทางเลือก | ตัดสินใจ | เหตุผล |
|---|---|---|---|
| ไฟล์มี BOM | อ่านด้วย utf-8-sig / แก้ไฟล์ | อ่านด้วย `utf-8-sig` | ไม่ต้องแก้ไฟล์ต้นฉบับ และใช้วิธีเดียวกับ sales_raw.csv |
| เบอร์โทรซ้ำ (ปิดบัง) | ถือว่าเป็นคนเดียวกัน / ไม่ใช้เบอร์เป็นกุญแจ | ไม่ใช้เบอร์เป็นกุญแจ ใช้ `customer_id` | จำนวนที่ชนใกล้ค่าที่คาดจากการสุ่ม และชื่อเล่น/เพศ/อายุไม่ตรงกันเลย จึงไม่ใช่สมาชิกซ้ำ |
| home_branch_id เป็นรหัส ไม่ตรงกับชื่อสาขาใน sales | ใช้ตาราง branches (รหัส↔ชื่อ) / เดา mapping | ใช้ตาราง `branches` ทางการ | mapping ที่อนุมานได้ตรงกับตารางทางการทุกรหัส แต่ควรอ้างอิงแหล่งทางการเสมอ |
| สมาชิกที่ไม่เคยซื้อ | ลบ / เก็บไว้ทำแคมเปญ | เก็บไว้ | เป็นสมาชิกจริง ใช้ทำแคมเปญกระตุ้นการซื้อครั้งแรกได้ แต่ KPI "สมาชิกที่ซื้อ" ต้องแยกจาก "สมาชิกทั้งหมด" |
| สาขาประจำ ≠ สาขาที่ซื้อบ่อย | แก้สาขาประจำ / เก็บไว้ทั้งคู่ | เก็บไว้ทั้งคู่ | สาขาประจำคือที่สมัคร ไม่ใช่ข้อผิดพลาด ส่วนสาขาที่ซื้อบ่อยคำนวณจาก sales ได้เสมอ |
| สมาชิกอายุต่ำกว่า 18 | เก็บตามเดิม / จำกัดการใช้ข้อมูล (PDPA) | เก็บไว้ แต่จำกัดการใช้ | ใช้ในสถิติรวมได้ ไม่ใช้ทำการตลาดเจาะจงตัวบุคคลโดยไม่มีความยินยอมจากผู้ปกครอง |
| nickname ซ้ำกันมาก | ใช้ระบุตัว / ไม่ใช้ระบุตัว | ไม่ใช้ระบุตัว | มีแค่ 24 ชื่อ สำหรับ 3,000 คน |

In [8]:
findings = pd.DataFrame([
    ("รูปแบบไฟล์", "มี BOM ต้นไฟล์", 1, "อ่านด้วย encoding='utf-8-sig'"),
    ("แถวซ้ำทุกคอลัมน์", "n_dup_rows", n_dup_rows, "ผ่าน" if n_dup_rows == 0 else "ต้องลบ"),
    ("customer_id ซ้ำ", "n_dup_id", n_dup_id, "ผ่าน · ใช้เป็นกุญแจได้" if n_dup_id == 0 else "ต้องตรวจ"),
    ("รูปแบบ customer_id", "n_bad_id", n_bad_id, "ผ่าน" if n_bad_id == 0 else "ต้องแก้"),
    ("joined_date", "n_bad_date", n_bad_date, "ผ่าน" if n_bad_date == 0 else "ต้องแก้"),
    ("รูปแบบ phone", "n_bad_phone", n_bad_phone, "ผ่าน" if n_bad_phone == 0 else "ต้องแก้"),
    ("เบอร์โทรซ้ำ (แถว)", "n_shared_phone", n_shared_phone, f"ใกล้ค่าที่คาดจากการสุ่ม (~{expected_pairs:.0f} คู่) → ไม่ใช้เบอร์เป็นกุญแจ"),
    ("id ใน sales ไม่มีใน customers", "n_orphan_sales", n_orphan_sales, "ผ่าน" if n_orphan_sales == 0 else "ต้องตรวจ"),
    ("สมาชิกไม่เคยซื้อ", "n_never_bought", n_never_bought, f"{n_never_bought / len(cust):.1%} ของสมาชิก · {recent} คนเพิ่งสมัคร"),
    ("สมัครหลังซื้อครั้งแรก", "n_joined_after_buy", n_joined_after_buy, "ผ่าน" if n_joined_after_buy == 0 else "ต้องตรวจ"),
    ("home_branch_id เป็นรหัส", "mapping", len(BRANCH_CODE),
     ("ตรงกับตาราง branches ทางการ: " if branches is not None else "อนุมานจากข้อมูล: ") + " · ".join(f"{k}={v}" for k, v in BRANCH_CODE.items())),
    ("สมัครก่อนสาขาประจำเปิด", "n_joined_before_open", n_joined_before_open, "ผ่าน" if n_joined_before_open == 0 else "ต้องตรวจ"),
    ("สาขาประจำ ≠ สาขาที่ซื้อบ่อย", "n_home_mismatch", n_home_mismatch, f"{n_home_mismatch / len(fav):.1%} ของคนที่เคยซื้อ"),
    ("สมาชิกอายุต่ำกว่า 18", "n_minor", n_minor, "ข้อมูลผู้เยาว์ ต้องระวังตาม PDPA"),
], columns=["หัวข้อ", "ตัวแปร", "จำนวน", "ข้อสังเกต"])
display(findings) if "display" in globals() else print(findings.to_string(index=False))

findings.to_csv("customers_profile.csv", index=False, encoding="utf-8-sig")
print("\nบันทึก customers_profile.csv แล้ว (ข้อมูล cust ไม่ถูกแก้)")
try:
    from google.colab import files
    files.download("customers_profile.csv")
except ImportError:
    pass

                       หัวข้อ               ตัวแปร  จำนวน                                                                                   ข้อสังเกต
                   รูปแบบไฟล์       มี BOM ต้นไฟล์      1                                                               อ่านด้วย encoding='utf-8-sig'
             แถวซ้ำทุกคอลัมน์           n_dup_rows      0                                                                                        ผ่าน
              customer_id ซ้ำ             n_dup_id      0                                                                      ผ่าน · ใช้เป็นกุญแจได้
           รูปแบบ customer_id             n_bad_id      0                                                                                        ผ่าน
                  joined_date           n_bad_date      0                                                                                        ผ่าน
                 รูปแบบ phone          n_bad_phone      0                                           

**คำถามชวนคิด**
1. ถ้าจะทำ Dashboard "ยอดขายตามช่วงอายุ" ต้อง join `sales` กับ `customers` ด้วยคอลัมน์อะไร ลูกค้าทั่วไป (customer_id ว่าง) จะหายไปจากกราฟไหม
2. สมาชิกที่ไม่เคยซื้อควรนับรวมใน KPI "จำนวนสมาชิก" หรือไม่ ถ้านับ ตัวเลขจะต่างจาก KPI ใน Dashboard (2,507 คนที่เคยซื้อ) อย่างไร
3. ทำไมไม่ควรเดา mapping รหัสสาขาจากข้อมูล แม้จะตรงกันถึง ~95% แล้วควรขอข้อมูลอะไรจากร้านเพิ่ม